# GOES GLM lightning over True Color

The Geostationary Lightning Mapper reports individual optical **flashes** with a
latitude, longitude and time. Here they are drawn on top of a True Color image.

The example is a line of afternoon thunderstorms over the central United States
on **3 October 2023, 20:00 UTC** (about 14:00 local, so there is daylight for
True Color). GOES-16 is used because that region is in the eastern satellite's
view — for GOES-18 change `BUCKET` and the domain.

Everything is in this notebook: the folders, the domain, the flash window and
the whole plotting code.


## Setup

In [ ]:
import glob
import math
import warnings
from pathlib import Path

import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from pyresample import create_area_def
from satpy import Scene
from satpy.enhancements.enhancer import get_enhanced_image
from shapely.geometry import box

warnings.filterwarnings("ignore")
import sys
from datetime import datetime

import xarray as xr

sys.path.insert(0, "..")
from examples.glm import abi_scan_keys, download_abi, download_lcfa, lcfa_keys


In [ ]:
# ---------------------------------------------------------------
# Helper kept in the notebook so you can change it: coastline segments
# clipped to a lon/lat box. Change "10m" to "50m" for a coarser, faster line.
# ---------------------------------------------------------------
def coastlines(bbox, resolution="10m"):
    west, south, east, north = bbox[0], bbox[1], bbox[2], bbox[3]
    clip = box(west, south, east, north)
    segments = []
    feature = cfeature.NaturalEarthFeature("physical", "coastline", resolution)
    for geometry in feature.intersecting_geometries([west, east, south, north]):
        piece = geometry.intersection(clip)
        for line in (piece.geoms if hasattr(piece, "geoms") else [piece]):
            if not line.is_empty and line.length > 0:
                x, y = line.xy
                segments.append((np.asarray(x), np.asarray(y)))
    return segments


def lon_label(value, _pos=None):
    v = ((value + 180) % 360) - 180
    return f"{abs(v):g}\u00b0{'E' if v >= 0 else 'W'}"


def lat_label(value, _pos=None):
    return f"{abs(value):g}\u00b0{'N' if value >= 0 else 'S'}"


In [ ]:
# ---------------------------------------------------------------
# STYLE -- change anything here and re-run the plot cells
# ---------------------------------------------------------------
COAST_COLOUR = "red"        # coastline colour
COAST_WIDTH = 0.8           # coastline line width
COAST_RES = "10m"           # "10m", "50m" or "110m"

GRID_COLOUR = "white"       # graticule colour
GRID_ALPHA = 0.45
GRID_STYLE = "--"
GRID_WIDTH = 0.5

MARKER_LON, MARKER_LAT = None, None   # no volcano in this scene
MARKER_COLOUR = "red"
MARKER_SIZE = 10

FIG_WIDTH = 13.5            # inches
DPI = 160

FLASH_COLOUR = "yellow"     # GLM flash marker
FLASH_SIZE = 14
FLASH_EDGE = 0.8


## 1. Your files

Two products for the same moment: the ABI channels for True Color, and the GLM
flashes. Point `ABI_DIR` and `GLM_DIR` at your own folders; the download block
only runs if they are empty.


In [ ]:
BUCKET = "noaa-goes16"
WHEN = datetime(2023, 10, 3, 20, 0)
FLASH_MINUTES = 10                 # how long a window of flashes to draw

ABI_DIR = Path("../data/glm-demo/abi16f")
GLM_DIR = Path("../data/glm-demo/glm16")

abi_files = sorted(glob.glob(str(ABI_DIR / "*.nc")))
if not abi_files:
    ABI_DIR.mkdir(parents=True, exist_ok=True)
    keys = abi_scan_keys(BUCKET, "ABI-L1b-RadF", WHEN, ("C01", "C02", "C03"))
    abi_files = sorted(download_abi(BUCKET, keys, ABI_DIR))

glm_files = sorted(glob.glob(str(GLM_DIR / "*.nc")))
if not glm_files:
    GLM_DIR.mkdir(parents=True, exist_ok=True)
    glm_files = download_lcfa(BUCKET, lcfa_keys(BUCKET, WHEN, minutes=FLASH_MINUTES), GLM_DIR)

print(f"ABI files: {len(abi_files)}")
print(f"GLM files: {len(glm_files)}  ({FLASH_MINUTES} minutes)")


## 2. Your domain

Write the four numbers yourself. GLM sees the whole disk, so this box decides
which flashes are drawn.


In [ ]:
DOMAIN = (-104.0, 36.0, -97.0, 43.0)   # min_lon, min_lat, max_lon, max_lat
RESOLUTION = 0.01

print("domain:", DOMAIN)


## 3. Read the flashes

Each LCFA file holds `flash_lat` / `flash_lon`. This is plain xarray, so you can
also filter on `flash_energy` or `flash_time_offset_of_first_event`.


In [ ]:
lons, lats = [], []
for name in glm_files:
    with xr.open_dataset(name) as ds:
        if "flash_lat" in ds:
            lats.append(np.asarray(ds["flash_lat"].values, dtype="float64"))
            lons.append(np.asarray(ds["flash_lon"].values, dtype="float64"))

lon = np.concatenate(lons) if lons else np.array([])
lat = np.concatenate(lats) if lats else np.array([])

west, south, east, north = DOMAIN
keep = (lon >= west) & (lon <= east) & (lat >= south) & (lat <= north)
flash_lon, flash_lat = lon[keep], lat[keep]
print(f"{len(lon)} flashes total, {len(flash_lon)} inside the domain")


## 4. Build True Color and crop to the domain

In [ ]:
COMPOSITE = "true_color"
TITLE = "True Color + GLM flashes"
OUT = Path("../output/glm_true_color.png")
OUT.parent.mkdir(parents=True, exist_ok=True)

scene = Scene(reader="abi_l1b", filenames=abi_files)
scene.load([COMPOSITE], generate=True)

area = create_area_def(
    "domain", {"proj": "longlat", "datum": "WGS84"},
    area_extent=DOMAIN, resolution=(RESOLUTION, RESOLUTION), units="degrees",
)
local = scene.resample(area)
picture = get_enhanced_image(local[COMPOSITE]).pil_image()
print("composite ready")


## 5. Plot — image plus the flashes on top

All the drawing is here. The flashes are a plain `scatter`, so change the marker,
colour or size freely.


In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_WIDTH * (north - south) / (east - west) * 0.92))

ax.imshow(picture, extent=(west, east, south, north), origin="upper",
          aspect="auto", zorder=0)

ax.grid(color=GRID_COLOUR, alpha=GRID_ALPHA, ls=GRID_STYLE, lw=GRID_WIDTH, zorder=1)

for line_x, line_y in coastlines(DOMAIN, COAST_RES):
    ax.plot(line_x, line_y, color=COAST_COLOUR, lw=COAST_WIDTH, zorder=4)

ax.scatter(flash_lon, flash_lat, s=FLASH_SIZE, marker="o",
           facecolors="none", edgecolors=FLASH_COLOUR, linewidths=FLASH_EDGE,
           zorder=5, label=f"GLM flashes ({len(flash_lon)})")
ax.legend(loc="upper right", framealpha=0.7, fontsize=9)

ax.set_xlim(west, east); ax.set_ylim(south, north)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lon_label))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lat_label))
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("G16 - " + TITLE, loc="left")
ax.set_title(f"{local[COMPOSITE].attrs['start_time']:%Y/%m/%d %H:%M} UTC", loc="right")

fig.tight_layout()
fig.savefig(OUT, dpi=DPI)
plt.close(fig)
display(Image(filename=str(OUT)))


## Notes

* GLM sees the flash at **cloud top**, so points sit over the anvil rather than
  at the ground strike point. That they land on the bright convective tops is the
  check that the geolocation lines up.
* `FLASH_MINUTES` sets how much of the storm's history is drawn in one frame.
* Detection efficiency drops towards the edge of the disk and at high latitude,
  which matters for volcanic lightning at Alaskan latitudes.
